## Ideia

Separar uma parte para validação; \
Há rodadas que representam o número de treinos/etapas.

### K-Fold

O dataset é aleatoriamente dividido em k subconjuntos de tamanhos iguais (chamados de folds). \
O modelo é treinado K-1 folds e validado no fold restante. \
Cada fold fica com $$n  \ amostras / k$$ amostras. \
Quando n não é múltiplo de k, faz a divisão inteira $n \ // k$ (quantidade de folds) e depois faz n \ % k (amostras que sobraram), inserindo cada amostra nos folds da esquerda pra direita, um por um.

### Stratified K-Fold

Existe a possibilidade de que em um conjunto de 100 amostras, 10 positivas fiquem em um fold e nenhuma nos outros, o que é um problema de desbalanceamento. \
Dessa forma, é preciso preservar a proporção das classes em cada fold.

### Quando usar cada

K-Fold é usado para regressão e Stratified para classificação (padrão do cross-val-score do sklearn).

In [2]:
import numpy as np
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score)
from sklearn.neighbors import NearestCentroid

#### Dados e Classes

In [3]:
rng = np.random.default_rng(0)
X0 = rng.normal(loc=[0,0], scale=1.0, size=(90, 2)) # Classe 0
X1 = rng.normal(loc=[2,2], scale=1.0, size=(10,2)) # Classe 1

X = np.vstack([X0, X1])
y = np.concatenate([np.zeros(90, dtype=int), np.ones(10, dtype=int)])

print(X.shape, y.shape)
print(np.unique(y, return_counts=True))

(100, 2) (100,)
(array([0, 1]), array([90, 10]))


In [ ]:
def meu_kfold(n_amostras, k=5, shuffle=False, random_state=None):
    """Gera K pares (train_idx, val_idx) cortando posições seguidas."""
    # 1 validar K
    if not 2 <= k <= n_amostras:
        raise ValueError("k precisa estar entre 2 e n_amostras")
    # 2 criar a sequência de índices
    indices = np.arange(n_amostras)
    # 3 embaralhar (opcional) — embaralhamos ÍNDICES, nunca X e y separadamente
    if shuffle:
        rng = np.random.default_rng(random_state)
        indices = rng.permutation(indices)
    # 4 tamanho de cada fold: base n // k; os primeiros n % k folds ganham +1
    tamanhos = np.full(k, n_amostras // k)
    tamanhos[: n_amostras % k] += 1
    inicio = 0
    # 5 percorrer os folds
    for tamanho in tamanhos:
        fim = inicio + tamanho
        # 6 fold de validação
        val_idx = indices[inicio:fim]
        # 7 treino = todo o resto
        train_idx = np.concatenate([indices[:inicio], indices[fim:]])
        # 8 entregar o par e pausar
        yield train_idx, val_idx
        inicio = fim   # sem isso, o primeiro fold se repetiria K vezes

In [8]:
def meu_stratified_kfold(y, k=5, shuffle=False, random_state=None):
    """Gera K pares (train_idx, val_idx) preservando a proporção das classes."""
    y = np.asarray(y)
    classes, contagens = np.unique(y, return_counts=True)        # 1
    if k < 2 or contagens.min() < k:
        raise ValueError("cada classe precisa ter pelo menos k amostras")
    rng = np.random.default_rng(random_state)
    partes_por_fold = [[] for _ in range(k)]                     # 2
    for c in classes:
        idx_c = np.flatnonzero(y == c)                           # 3 posições da classe c
        if shuffle:
            idx_c = rng.permutation(idx_c)                       #   embaralha dentro da classe
        for f, parte in enumerate(np.array_split(idx_c, k)):    # 4 parte f -> fold f
            partes_por_fold[f].append(parte)
    todos = np.arange(len(y))
    for partes in partes_por_fold:
        val_idx = np.sort(np.concatenate(partes))                # 5 validação
        train_idx = np.setdiff1d(todos, val_idx)                 # 6 treino = complemento
        yield train_idx, val_idx                                 # 7

In [9]:
def contar_classes(gerador):
    for fold, (train_idx, val_idx) in enumerate(gerador, start=1):
        print(fold, np.bincount(y[val_idx], minlength=2))

contar_classes(meu_kfold(100, k=5))
contar_classes(meu_kfold(100, k=5, shuffle=True), random_state=42)
contar_classes(meu_stratified_kfold(y, k=5, shuffle=True, random_state=42))

TypeError: 'NoneType' object is not iterable

In [ ]:
class CentroideMaisProximo:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroides_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        return self
    
    def predict(self, X):
        dist = np.stack([np.linalg.norm(X - c, axis = 1) for c in self.centroides_], axis=1)
        return self.classes_[np.argmin(dist, axis=1)]

def acuracia(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [10]:
def meu_cross_val_score(modelo_factory, X, y, folds, metrica):
    scores = []
    for train_idx, val_idx in folds:
        X_treino, y_treino = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        modelo = modelo_factory()            # novo a cada rodada
        modelo.fit(X_treino, y_treino)
        y_pred = modelo.predict(X_val)
        scores.append(metrica(y_val, y_pred))
    return np.array(scores)

In [ ]:
folds = meu_stratified_kfold(y, k=5, shuffle=True, random_state=42)
scores = meu_cross_val_score(CentroideMaisProximo, X, y, folds, acuracia)
print(scores, f"{scores.mean():.3f} +- {scores.std():.3f}")

In [ ]:
all(np.array_equal(v1,v2) for (_, v1), (_, v2) in zip(meu_kfold(100, k=5), KFold(5).split(X)))

meus_folds = list(meu_stratified_kfold(y, k=5, shuffle=True, random_state=42))
cross_val_score(NearestCentroid(), X, y, cv=meus_folds)

cv = StratifiedKFold(5, shuffle=True, random_state=42)
cross_val_score(NearestCentroid(), X, y, cv=cv)